In [20]:
import time
import pandas as pd
import uuid
import urllib.parse
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import TimeoutException

def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--lang=id') 
    options.add_argument('--start-maximized') 
    # options.add_argument('--headless') 
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver

# FITUR BARU: Fungsi untuk menyimpan data secara berkala (Checkpoint)
def save_checkpoint(wisata_data, rating_data):
    if wisata_data:
        pd.DataFrame(wisata_data).to_csv("tabel_wisata.csv", index=False)
    if rating_data:
        pd.DataFrame(rating_data).to_csv("tabel_rating.csv", index=False)

def scrape_google_maps(queries):
    driver = setup_driver()
    wisata_data = []
    rating_data = []

    for idx, query in enumerate(queries, start=1):
        if not query.strip():  
            continue
            
        print(f"\n[{idx}/{len(queries)}] Sedang memproses: {query}...")
        
        encoded_query = urllib.parse.quote(query)
        url_pencarian = f"https://www.google.com/maps/search/{encoded_query}?hl=id"
        
        sukses_scraping = False 
        
        for attempt in range(2):
            if sukses_scraping:
                break 
                
            if attempt > 0:
                print(f"-> Mencoba ulang {query} (Percobaan ke-{attempt + 1})...")
                
            driver.get(url_pencarian)
            wait = WebDriverWait(driver, 15)
            
            try:
                # Bypass List View / Deteksi Dinamis
                wait.until(EC.presence_of_element_located((
                    By.XPATH, '//h1[contains(@class, "DUwDvf")] | //a[contains(@href, "/maps/place/")]'
                )))
                time.sleep(2) 
                
                elemen_list = driver.find_elements(By.XPATH, '//a[contains(@href, "/maps/place/")]/ancestor::div[contains(@class, "Nv2PK")]')
                
                if elemen_list:
                    for item in elemen_list:
                        teks_item = item.text.lower()
                        if "bersponsor" not in teks_item and "sponsored" not in teks_item:
                            link = item.find_element(By.XPATH, './/a[contains(@href, "/maps/place/")]')
                            driver.execute_script("arguments[0].click();", link) 
                            break
                    time.sleep(3) 
                
                # Ekstraksi Judul
                judul_elemen = wait.until(EC.presence_of_element_located((By.XPATH, '//h1[contains(@class, "DUwDvf")] | //h1')))
                nama_wisata = judul_elemen.text
                
                if "bersponsor" in nama_wisata.lower() or not nama_wisata:
                    raise Exception("Tertangkap elemen bersponsor.")
                
                id_wisata = "W_" + str(uuid.uuid4())[:8]
                kategori = "Alam" if any(k in query.lower() for k in ["pantai", "gunung", "air terjun", "goa", "bukit"]) else "Budaya/Sejarah"
                
                # Ekstraksi Alamat
                try:
                    alamat_elemen = driver.find_element(By.XPATH, '//button[@data-item-id="address"]')
                    alamat_lengkap = alamat_elemen.text.lower()
                except:
                    alamat_lengkap = ""

                teks_analisis = alamat_lengkap + " " + query.lower() + " " + nama_wisata.lower()
                keyword_jogja = ['yogyakarta', 'jogja', 'bantul', 'sleman', 'gunungkidul', 'gunung kidul', 'kulon progo', 'kulonprogo', 'diy']
                keyword_solo = ['solo', 'surakarta', 'karanganyar', 'sukoharjo', 'boyolali', 'klaten', 'sragen', 'wonogiri']

                if any(k in teks_analisis for k in keyword_jogja):
                    kota = "Yogyakarta"
                elif any(k in teks_analisis for k in keyword_solo):
                    kota = "Solo"
                else:
                    kota = "Tidak Terdeteksi" 
                
                # Ekstraksi Deskripsi
                try:
                    deskripsi = driver.find_element(By.XPATH, '//div[contains(@class, "PYvSYb")]').text
                except:
                    deskripsi = f"Destinasi wisata {nama_wisata} berlokasi di {kota}."

                # Ekstraksi Ulasan
                try:
                    tab_ulasan = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[.//div[contains(text(), "Ulasan")]] | //button[contains(@aria-label, "Ulasan")] | //button[.//div[contains(text(), "Reviews")]]')))
                    driver.execute_script("arguments[0].click();", tab_ulasan)
                    time.sleep(3) 
                    
                    wisata_data.append({
                        "ID_Wisata": id_wisata,
                        "Nama_Wisata": nama_wisata,
                        "Kota": kota,
                        "Kategori": kategori,
                        "Deskripsi_Wisata": deskripsi
                    })
                    print(f"-> Berhasil mendapatkan info: {nama_wisata} ({kota})")
                    
                    scrollable_div = wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "m6QErb DxyBCb")]')))
                    
                    print("-> Mengambil ulasan...")
                    for _ in range(25): 
                        driver.execute_script('arguments[0].scrollTop = arguments[0].scrollHeight', scrollable_div)
                        time.sleep(2)

                    reviews = driver.find_elements(By.XPATH, '//div[contains(@class, "jJc9Ad")]')
                    for review in reviews:
                        try:
                            nama_user = review.find_element(By.XPATH, './/div[contains(@class, "d4r55")]').text
                            id_user = "U_" + str(hash(nama_user))[-6:] 
                            
                            rating_element = review.find_element(By.XPATH, './/span[contains(@class, "kvMYJc")]')
                            rating_text = rating_element.get_attribute("aria-label")
                            rating_angka = int(rating_text.split(" ")[0])

                            rating_data.append({
                                "ID_User": id_user,
                                "ID_Wisata": id_wisata,
                                "Rating": rating_angka
                            })
                        except Exception:
                            continue 
                    
                    sukses_scraping = True 
                    
                    # --- EKSEKUSI CHECKPOINT ---
                    # File disave setiap kali 1 destinasi selesai di-scrape dengan sukses!
                    save_checkpoint(wisata_data, rating_data)
                    print(f"-> [CHECKPOINT] Data {nama_wisata} berhasil diamankan ke CSV.")

                except Exception as e:
                    print("-> Tab ulasan tidak ditemukan.")
                    if attempt == 1: 
                        print(f"-> Gagal mengekstrak ulasan untuk {query} setelah 2 percobaan.")

            except Exception as e:
                print(f"-> Gagal memuat detail utama {query}.")
                if attempt == 1:
                    print(f"-> Melewati {query} sepenuhnya.")

    driver.quit()
    
    print("\n" + "=" * 50)
    print(f"PROSES SCRAPING SELESAI!")
    print(f"Total tempat wisata tersimpan: {len(wisata_data)}")
    print(f"Total ulasan tersimpan: {len(rating_data)}")
    print("=" * 50)

# Masukkan daftar tempat wisata kamu di sini
daftar_destinasi = [
    # === DESTINASI YOGYAKARTA (1 - 100) ===
    "Jl. Malioboro", 
    "Keraton Ngayogyakarta Hadiningrat",
    "Kampung Wisata Taman Sari",
    "Tugu Jogja",
    "Museum Benteng Vredeburg",
    "Candi Prambanan",
    "Situs Ratu Boko",
    "Candi Sewu",
    "Candi Sambisari",
    "Candi Ijo",
    "Candi Plaosan",
    "Monumen Jogja Kembali",
    "Pantai Parangtritis",
    "Pantai Indrayanti",
    "Pantai Timang",
    "Pantai Jogan",
    "Pantai Pok Tunggal",
    "Pantai Nglambor",
    "Pantai Drini",
    "Pantai Glagah",
    "Hutan Pinus Mangunan Dlingo",
    "Puncak Becici",
    "Kebun Buah Mangunan",
    "Gunung Api Purba Nglanggeran",
    "Bunker Kaliadem Merapi",
    "Goa Pindul",
    "Goa Jomblang",
    "Lava Tour Merapi",
    "Taman Pelangi Jogja",
    "HeHa Sky View",
    "HeHa Ocean View",
    "Obelix Hills",
    "The Lost World Castle",
    "Taman Tebing Breksi",
    "Gembira Loka Zoo",
    "Blue Lagoon Jogja",
    "Puncak Sosok",
    "Kawasan Titik Nol Kilometer Yogyakarta",
    "Taman Pintar Yogyakarta",
    "Museum Sonobudoyo",
    "Pasar Beringharjo",
    "Museum Ullen Sentalu",
    "Sindu Kusuma Edupark",
    "Embung Nglanggeran",
    "Hutan Pinus Pengger",
    "Pintu Langit Dahromo",
    "Bukit Bintang",
    "Ekowisata Taman Sungai Mudal",
    "Waduk Sermo",
    "De Mata Trick Eye Museum",
    "Alun-Alun Kidul Yogyakarta",
    "Kids Fun Parcs",
    "Candi Kalasan",
    "Candi Sari",
    "Candi Barong",
    "Candi Banyunibo",
    "Air Terjun Sri Gethuk",
    "Pantai Sadranan",
    "Pantai Krakal",
    "Pantai Kukup",
    "Pantai Baron",
    "Pantai Sepanjang",
    "Pantai Ngandong",
    "Pantai Wediombo",
    "Pantai Siung",
    "Pantai Ngobaran",
    "Pantai Ngrenehan",
    "Pantai Gesing",
    "Pantai Kuwaru",
    "Gumuk Pasir Parangkusumo",
    "Hutan Pinus Asri",
    "Seribu Batu Songgo Langit",
    "Jurang Tembelan Kanigoro",
    "Bukit Pengilon",
    "Bukit Paralayang Watugupit",
    "Grojogan Watu Purbo",
    "Air Terjun Kedung Pedut",
    "Kembang Soka Waterfall",
    "Pule Payung",
    "Kalibiru National Park",
    "Ayunan Langit Watu Jaran",
    "Goa Kiskendo",
    "Goa Rancang Kencono",
    "Museum Affandi",
    "Museum Dirgantara Mandala",
    "Museum Gunung Merapi",
    "Merapi Park Yogyakarta",
    "Agrowisata Bhumi Merapi",
    "Stonehenge Merapi",
    "Plunyon Kalikuning",
    "Bukit Kali Kuning",
    "Waterboom Jogja",
    "Jogja Exotarium",
    "Suraloka Zoo",
    "Watu Goyang",
    "Puncak Segoro",
    "Teras Kaca Pantai Nguluran",
    "Studio Alam Gamplong",
    "Tebing Gunung Gajah",
    "Pantai Jungwok",

    # === DESTINASI SURAKARTA & SEKITARNYA (101 - 200) ===
    "Keraton Surakarta Hadiningrat",
    "Pura Mangkunegaran",
    "Museum Radya Pustaka",
    "Museum Batik Danar Hadi",
    "Benteng Vastenburg",
    "Museum Keris Nusantara",
    "Monumen Pers Nasional",
    "Lokananta Bloc",
    "Gedung Djoeang '45 Solo",
    "Ndalem Kalitan",
    "Tumurun Private Museum",
    "Kampung Batik Laweyan",
    "Kampung Wisata Batik Kauman",
    "Gedung Wayang Orang Sriwedari",
    "Taman Sriwedari Surakarta",
    "Masjid Raya Sheikh Zayed Solo",
    "Masjid Agung Surakarta",
    "Masjid Al Wustho Mangkunegaran",
    "Vihara Dhamma Sundara",
    "Gereja Katolik Santo Antonius Purbayan",
    "Solo Safari",
    "Taman Balekambang Surakarta",
    "Pracima Tuin",
    "Bendung Karet Tirtonadi",
    "Taman Cerdas Jebres",
    "Alun-Alun Kidul Surakarta",
    "Alun-Alun Lor Surakarta",
    "Taman Jayawijaya Mojosongo",
    "Monumen 45 Banjarsari",
    "Taman Sunan Jogo Kali",
    "Pasar Barang Antik Triwindu",
    "Pasar Gede Hardjonagoro",
    "Pasar Klewer Surakarta",
    "Ngarsopuro Night Market",
    "Stadion Manahan",
    "Koridor Gatsu Solo",
    "Tugu Kebangkitan Nasional (Tugu Lilin)",
    "Plaza Manahan",
    "Stasiun Solo Balapan",
    "Tugu Makutha Solo",
    "De' Tjolomadoe",
    "The Heritage Palace",
    "Pandawa Water World",
    "Grojogan Sewu Tawangmangu",
    "Candi Cetho",
    "Candi Sukuh",
    "The Lawu Park",
    "Umbul Ponggok",
    "Waduk Cengklik Park",
    "Kemuning Sky Hills",
    "Air Terjun Jumog",
    "Telaga Madirda",
    "Kebun Teh Kemuning",
    "Bukit Mongkrang",
    "Sakura Hills Tawangmangu",
    "Bukit Sekipan Tawangmangu",
    "Taman Balekambang Tawangmangu",
    "Rumah Atsiri Indonesia",
    "Umbul Manten",
    "Umbul Cokro",
    "Umbul Sigedang",
    "Umbul Pelem Waterpark",
    "Umbul Besuki",
    "Candi Merak",
    "Rowo Jombor",
    "Bukit Sidoguro",
    "Air Terjun Girimanik",
    "Waduk Gajah Mungkur",
    "Museum Manusia Purba Sangiran",
    "Gunung Kemukus",
    "Waduk Kedung Ombo",
    "Alun-Alun Karanganyar",
    "Alun-Alun Boyolali",
    "Simpang Lima Boyolali",
    "Monumen Susu Murni Boyolali",
    "Cepogo Cheese Park",
    "New Selo",
    "Bukit Sanjaya Selo",
    "Kebun Raya Indrokilo Boyolali",
    "Obyek Wisata Mata Air Pengging",
    "Bendungan Colo",
    "Gunung Sepikul",
    "Batu Seribu",
    "Bukit Hope Karanganyar",
    "Museum Samanhoedi",
    "Petilasan Keraton Kartasura",
    "Kali Pucung Kemuning",
    "Agrowisata Sondokoro",
    "Candi Menggung",
    "Pemandian Air Panas Bayanan",
    "Air Terjun Parang Ijo",
    "Taman Hutan Raya KGPAA Mangkunagoro I",
    "Astana Giribangun",
    "Telaga Claket",
    "Bukit Gancik Selo",
    "Air Terjun Kedung Sriti",
    "Soko Alas Ponggok",
    "Janti Park",
    "Umbul Nilo",
    "Deles Indah"
]

scrape_google_maps(daftar_destinasi)


[1/200] Sedang memproses: Jl. Malioboro...
-> Tab ulasan tidak ditemukan.
-> Mencoba ulang Jl. Malioboro (Percobaan ke-2)...
-> Berhasil mendapatkan info: Jl. Malioboro (Tidak Terdeteksi)
-> Mengambil ulasan...
-> [CHECKPOINT] Data Jl. Malioboro berhasil diamankan ke CSV.

[2/200] Sedang memproses: Keraton Ngayogyakarta Hadiningrat...
-> Berhasil mendapatkan info: Keraton Ngayogyakarta Hadiningrat (Yogyakarta)
-> Mengambil ulasan...
-> [CHECKPOINT] Data Keraton Ngayogyakarta Hadiningrat berhasil diamankan ke CSV.

[3/200] Sedang memproses: Kampung Wisata Taman Sari...
-> Berhasil mendapatkan info: Kampung Wisata Taman Sari (Tidak Terdeteksi)
-> Mengambil ulasan...
-> [CHECKPOINT] Data Kampung Wisata Taman Sari berhasil diamankan ke CSV.

[4/200] Sedang memproses: Tugu Jogja...
-> Berhasil mendapatkan info: Tugu Jogja (Yogyakarta)
-> Mengambil ulasan...
-> [CHECKPOINT] Data Tugu Jogja berhasil diamankan ke CSV.

[5/200] Sedang memproses: Museum Benteng Vredeburg...
-> Berhasil mendapat